# Pruning + QAT + Knowledge Distillation — ECG (Normal / Anómalo)
### Versión Google Colab — Taller PyCon

Se toma el modelo **Teacher** entrenado en P1 y se comprime en dos etapas: primero **Pruning** (poda) + **QAT** (cuantización consciente del entrenamiento), y luego se destila ese conocimiento en un modelo **Student** mucho más pequeño, listo para FPGA.

**Dataset:** PTB Diagnostic ECG — 187 puntos por latido.

**Nota:** al igual que en P1, este notebook corre en su propia VM de Colab. Necesita: (1) los CSVs de datos (se regeneran abajo, igual que en P1), y (2) **el modelo Teacher guardado en P1** — como Colab no comparte archivos entre notebooks, tendrás que subirlo manualmente la primera vez (celda marcada más abajo).

### Instalación de librerías (Colab)

In [1]:
!pip install -q qkeras tensorflow-model-optimization gdown

# Cuando tengas el repo listo, reemplaza por:
# !git clone https://github.com/tu-usuario/taller-ecg-fpga.git
# %cd taller-ecg-fpga
# (asegurate de que src/distillationClassKeras.py este ahi)

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.8/152.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.7 MB/s eta 0:00:00


### Preparación de datos (versión condensada de P0 — igual que en P1)

In [2]:
USE_DRIVE = False   # True = Google Drive (gdown) | False = carga manual

FILE_ID = 'REEMPLAZAR_CON_TU_FILE_ID'   # solo si USE_DRIVE = True

import zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import shutil

if USE_DRIVE:
    import gdown
    gdown.download(id=FILE_ID, output='ptbdb_ecg.zip', quiet=False)
else:
    from google.colab import files
    print('Sube tu archivo ptbdb_ecg.zip:')
    uploaded = files.upload()
    assert len(uploaded) > 0, 'No subiste ningun archivo'
    uploaded_name = list(uploaded.keys())[0]
    shutil.move(uploaded_name, 'ptbdb_ecg.zip')
    print(f"Archivo '{uploaded_name}' guardado como 'ptbdb_ecg.zip'")

data_dir = Path('dataset')
data_dir.mkdir(exist_ok=True)
with zipfile.ZipFile('ptbdb_ecg.zip', 'r') as z:
    z.extractall(data_dir)

LABEL_COL = 187
raw_normal   = pd.read_csv(data_dir / 'ptbdb_normal.csv',   header=None)
raw_abnormal = pd.read_csv(data_dir / 'ptbdb_abnormal.csv', header=None)

normal_vect   = raw_normal.iloc[:,   :LABEL_COL].values.astype(np.float32)
abnormal_vect = raw_abnormal.iloc[:, :LABEL_COL].values.astype(np.float32)

df_normal_full   = pd.DataFrame(normal_vect);   df_normal_full["class"]   = 0
df_abnormal_full = pd.DataFrame(abnormal_vect); df_abnormal_full["class"] = 1

n_samples = min(1000, len(df_normal_full), len(df_abnormal_full))
test_normal   = df_normal_full.sample(n=n_samples, random_state=45)
test_abnormal = df_abnormal_full.sample(n=n_samples, random_state=45)
train_normal   = df_normal_full.drop(test_normal.index)
train_abnormal = df_abnormal_full.drop(test_abnormal.index)

test_df  = pd.concat([test_normal, test_abnormal]).sample(frac=1, random_state=42).reset_index(drop=True)
train_df = pd.concat([train_normal, train_abnormal]).sample(frac=1, random_state=42).reset_index(drop=True)

test_df.to_csv("dataset/test_dataset.csv",   index=False)
train_df.to_csv("dataset/train_dataset.csv", index=False)

print(f"train_df: {train_df.shape}  |  test_df: {test_df.shape}")


Sube tu archivo ptbdb_ecg.zip:


Saving ptbdb_ecg.zip to ptbdb_ecg.zip
Archivo 'ptbdb_ecg.zip' guardado como 'ptbdb_ecg.zip'
train_df: (12552, 188)  |  test_df: (2000, 188)


### Librerías (pipeline ML)

**Importante:** `src.distillationClassKeras` contiene la clase `Distiller` que se usa más abajo para Knowledge Distillation. Si aún no tienes el repo, sube ese archivo manualmente a la sesión de Colab (icono de carpeta a la izquierda → arrastrar archivo) antes de correr esta celda.

In [ ]:
# Cargar el modulo de Knowledge Distillation (Distiller)
from pathlib import Path
import os

distill_path = Path('src/distillationClassKeras.py')
distill_path.parent.mkdir(exist_ok=True)

if not distill_path.exists():
    print('No se encontro src/distillationClassKeras.py en esta sesion.')
    print('Sube aqui ese archivo (el mismo que usabas en VS Code / el repo original):')
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        Path(fname).rename(distill_path)

# Crear src/__init__.py si no existe (necesario para que Python trate src/ como paquete)
init_path = Path('src/__init__.py')
if not init_path.exists():
    init_path.touch()

print('Modulo de distilacion listo en', distill_path)


No se encontro src/distillationClassKeras.py en esta sesion.
Sube aqui ese archivo (el mismo que usabas en VS Code / el repo original):


In [ ]:
import os
import numpy as np
from numpy import array
import matplotlib.pyplot as plt
import seaborn as sn
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.regularizers import l2, l1

import tensorflow_model_optimization as tfmot

# Pruning API
from tensorflow_model_optimization.python.core.sparsity.keras import prune, pruning_schedule, pruning_callbacks
from tensorflow_model_optimization.sparsity.keras import strip_pruning

# Quantization API
from qkeras import *

# Knowledge Distillation (requiere src/distillationClassKeras.py — ver nota arriba)
from src.distillationClassKeras import *

# Metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils import shuffle

# Pre-processing
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder

# Training utils
from sklearn.model_selection import train_test_split

tf.random.set_seed(42)
np.random.seed(42)

N_INPUT = 187   # EJ-299-33 usaba 161

### Función de preprocesamiento

Misma adaptación que en P1: sin normalización `/512.0` (PTB ya viene en [0,1]).

In [ ]:
def preproc_dataset(signal_dfAbnormal, signal_dfNormal,
                    train_size=2000,
                    test_size=500,
                    seed=42):

    LABEL = "class"

    signal_dfAbnormal = shuffle(signal_dfAbnormal, random_state=seed)
    signal_dfNormal   = shuffle(signal_dfNormal,   random_state=seed)

    max_available = min(len(signal_dfAbnormal), len(signal_dfNormal)) - test_size
    if train_size > max_available:
        print(f"Ajustando train_size de {train_size} a {max_available} (maximo disponible)")
        train_size = max_available

    train_abnormal = signal_dfAbnormal.iloc[:train_size]
    test_abnormal  = signal_dfAbnormal.iloc[train_size:train_size+test_size]

    train_normal = signal_dfNormal.iloc[:train_size]
    test_normal  = signal_dfNormal.iloc[train_size:train_size+test_size]

    dfTrain = pd.concat([train_abnormal, train_normal])
    dfTest  = pd.concat([test_abnormal,  test_normal])

    dfTrain = shuffle(dfTrain, random_state=seed).reset_index(drop=True)
    dfTest  = shuffle(dfTest,  random_state=seed).reset_index(drop=True)

    # PTB ya viene en [0,1] -> no se re-normaliza aqui
    return dfTrain, dfTest

### Carga de datos

In [ ]:
PATH = "dataset/"
TRAIN_DATASET_FILE = PATH + "train_dataset.csv"
TEST_DATASET_FILE  = PATH + "test_dataset.csv"

dfTrain = pd.read_csv(TRAIN_DATASET_FILE)
dfTestPTB = pd.read_csv(TEST_DATASET_FILE)

dfNormal   = dfTrain[dfTrain["class"] == 0]
dfAbnormal = dfTrain[dfTrain["class"] == 1]

### Preprocesamiento y separación de dataset

In [ ]:
df_train, dfTest = preproc_dataset(dfAbnormal, dfNormal)

df_train_ = df_train.pop('class')
dfTest_   = dfTest.pop('class')

le = LabelEncoder()
y = le.fit_transform(df_train_)
y = to_categorical(y, 2)

le = LabelEncoder()
yTest = le.fit_transform(dfTest_)
yTest = to_categorical(yTest, 2)

x_train, x_val, y_train, y_val = train_test_split(df_train, y, test_size=0.3, random_state=42)

x_train = x_train.astype(np.float32)
x_val   = x_val.astype(np.float32)

print(f"x_train: {x_train.shape} | x_val: {x_val.shape} | dfTest: {dfTest.shape}")

## Carga del modelo Teacher

**Paso manual la primera vez:** sube aquí el archivo `teacherModel_ECG_PTB.h5` que descargaste al final de P1 (panel izquierdo de Colab → arrastrar archivo, o usa el selector de abajo).

In [ ]:
from pathlib import Path

teacher_path = Path('models/teacherModel_ECG_PTB.h5')
teacher_path.parent.mkdir(exist_ok=True)

if not teacher_path.exists():
    print('No se encontro el modelo Teacher localmente.')
    print('Sube aqui el archivo teacherModel_ECG_PTB.h5 generado en P1:')
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        Path(fname).rename(teacher_path)

teacher_model = keras.models.load_model(str(teacher_path))
teacher_model.summary()

### Podado, Pruning

Se poda el modelo Teacher entrenado en P1.

In [ ]:
epochs = 100
lr = 0.001
loss = 'categorical_crossentropy'
op = Adam(learning_rate=lr)
metrics = ['accuracy']
batch = 64
val_split = 0.2

final_sparsity = 0.3

pruning_params = {
                 'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
                    initial_sparsity=0,
                    final_sparsity=final_sparsity,
                    begin_step=0,
                    end_step=1000
                 )
                 }

In [ ]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.4,
        patience=3,
        verbose=1
    ),
    pruning_callbacks.UpdatePruningStep()
]

### Podado del modelo

In [ ]:
modelP = tfmot.sparsity.keras.prune_low_magnitude(teacher_model, **pruning_params)
modelP.compile(optimizer=op, loss=loss, metrics=metrics)

In [ ]:
history_P = modelP.fit(x=x_train, y=y_train,
                       validation_split=val_split,
                       batch_size=batch,
                       epochs=epochs,
                       callbacks=callbacks,
                       verbose=1
                       )

In [ ]:
print(history_P.history.keys())

plt.figure(figsize=(15,3))
plt.plot(history_P.history['accuracy'])
plt.plot(history_P.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()
plt.figure()

plt.figure(figsize=(15,3))
plt.plot(history_P.history['loss'])
plt.plot(history_P.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

### Confusion matrix

Matriz de confusión después del podado

In [ ]:
y_pred_probs = modelP.predict(dfTest)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(yTest, axis=1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Anomalo', 'Normal'])
disp.plot(cmap="Blues")
plt.title('Confusion Matrix - Teacher model after pruning (ECG PTB)')
plt.show()

### Paso obligatorio después del pruning

Antes de exportar el modelo debes eliminar los wrappers:

In [ ]:
modelP = strip_pruning(modelP)

---

## Cuantización (QAT)

Aquí se aplicará la cuantización para reducir el tamaño y resolución de la red.

**Cambio vs. EJ-299-33:** `input_shape` pasa de 161 a `N_INPUT` (187). El resto de la arquitectura (bits de cuantización, capas) no cambia.

In [ ]:
## Estrategia de cuantizacion: 8 bits para kernel, bias y activacion

neurons_teacher = [10, 5, 7, 5, 6]

kernelQ      = "quantized_bits(8,4,alpha=1)"
biasQ        = "quantized_bits(8,4,alpha=1)"
activationQ  = "quantized_bits(8,4,alpha=1)"

modelQAT = Sequential(
    [
        QDense(neurons_teacher[0], name='input', input_shape=(N_INPUT,),
                           kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                           kernel_initializer='lecun_uniform'),
                    QActivation(activation=activationQ, name='relu0'),

                    QDense(neurons_teacher[1], name='fc1',
                           kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                           kernel_initializer='lecun_uniform'),
                    QActivation(activation=activationQ, name='relu1'),

                    QDense(neurons_teacher[2], name='fc2',
                           kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                           kernel_initializer='lecun_uniform'),
                    QActivation(activation=activationQ, name='relu2'),
                    Dropout(0.1),

                    QDense(neurons_teacher[3], name='fc3',
                           kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                           kernel_initializer='lecun_uniform'),
                    QActivation(activation=activationQ, name='relu3'),
                    Dropout(0.2),

                    QDense(neurons_teacher[4], name='fc4',
                           kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                           kernel_initializer='lecun_uniform'),
                    QActivation(activation=activationQ, name='relu4'),
                    Dropout(0.2),

                    QDense(2, name='output',
                           kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                           kernel_initializer='lecun_uniform'),
                    Activation(activation='sigmoid', name='sigmoid')
    ],
    name = "QuantizedModel",
)

In [ ]:
modelQAT.summary()

### Creación del modelo cuantizado

Aquí se realiza el entrenamiento

In [ ]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.4,
        patience=3,
        verbose=1
    ),
    pruning_callbacks.UpdatePruningStep()
]

In [ ]:
epochs = 150
lr = 0.0001
loss = 'binary_crossentropy'
op = Adam(learning_rate=lr)
metrics = ['accuracy']
batch = 32
val_split = 0.2

modelQAT.compile(optimizer=op, loss=loss, metrics=metrics)

history_QAT = modelQAT.fit(x=x_train, y=y_train,
                       validation_split=val_split,
                       epochs=epochs,
                       batch_size=batch,
                       callbacks=callbacks,
                       verbose=1
                       )

In [ ]:
print(history_QAT.history.keys())

plt.figure(figsize=(15,3))
plt.plot(history_QAT.history['accuracy'])
plt.plot(history_QAT.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()
plt.figure()

plt.figure(figsize=(15,3))
plt.plot(history_QAT.history['loss'])
plt.plot(history_QAT.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

In [ ]:
y_pred_probs = modelQAT.predict(dfTest)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(yTest, axis=1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Anomalo', 'Normal'])
disp.plot(cmap="Blues")
plt.title('Confusion Matrix - Teacher model obtained through QAT (ECG PTB)')
plt.show()

### Guardado del modelo podado y cuantizado

In [ ]:
modelQAT.save('models/teacherQATModel_ECG_PTB.h5')

---

# Entrenamiento del modelo Student

Para obtener el modelo reducido que será implementado en la FPGA, se emplea **Knowledge Distillation (KD)**: el modelo maestro (`modelQAT`) transfiere lo aprendido al modelo estudiante, que además se define aplicando cuantización y poda.

- 4-8 bits para la cuantización (según capa)
- 50% de sparsity

**Referencia QKeras:** Coelho et al. (2020). *Ultra low-latency, low-area inference accelerators using heterogeneous deep quantization with QKeras and hls4ml*. arXiv:2006.10159.

### Hiperparámetros para el modelo Student

In [ ]:
lr = 0.001
neurons_student = [6, 4, 2, 4, 3]

**Cambio vs. EJ-299-33:** `Input(shape=(161,))` → `Input(shape=(N_INPUT,))` (187). El resto de la arquitectura de cuantización es idéntica.

In [ ]:
def student_topology(neurons_student):
    kernelQ_4b = "quantized_bits(4,2,alpha=1)"
    biasQ_4b = "quantized_bits(4,2,alpha=1)"
    activationQ_4b = 'quantized_bits(4, 0)'

    kernelQ = "quantized_bits(8,4,alpha=1)"
    biasQ = "quantized_bits(8,4,alpha=1)"
    activationQ = 'quantized_bits(8,4)'

    kernelQ_16b = "quantized_bits(16,6,alpha=1)"
    biasQ_16b = "quantized_bits(16,6,alpha=1)"
    activationQ_16b = 'quantized_bits(16,6)'

    studentQ_MLP = keras.Sequential(
            [
                Input(shape=(N_INPUT,), name='inputLayer'),
                QDense(neurons_student[0], name='fc1',
                        kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                        kernel_initializer='lecun_uniform'),
                QActivation(activation=activationQ, name='relu0'),
                Dropout(0.1),

                QDense(neurons_student[1], name='fc2',
                        kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                        kernel_initializer='lecun_uniform'),
                QActivation(activation=activationQ, name='relu1'),
                Dropout(0.1),

                QDense(neurons_student[2], name='fc3',
                        kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                        kernel_initializer='lecun_uniform'),
                QActivation(activation=activationQ, name='relu2'),
                Dropout(0.1),

                QDense(neurons_student[3], name='fc4',
                        kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                        kernel_initializer='lecun_uniform'),
                QActivation(activation=activationQ, name='relu3'),
                Dropout(0.2),

                QDense(neurons_student[4], name='fc5',
                       kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                       kernel_initializer='lecun_uniform'),
                QActivation(activation=activationQ, name='relu4'),
                Dropout(0.2),

                QDense(2, name='output',
                        kernel_quantizer=kernelQ_16b, bias_quantizer=biasQ_16b,
                        kernel_initializer='lecun_uniform'),
                Activation(activation='sigmoid', name='outputActivation')
            ],
            name="studentMLP",
        )

    return studentQ_MLP

### Construcción del modelo

In [ ]:
def build_student(student_neurons, x_train, batch_size=64, epochs=200):
    qmodel = student_topology(student_neurons)

    steps_per_epoch = np.ceil(len(x_train) / batch_size)
    end_step = steps_per_epoch * epochs

    pruning_params = {
        "pruning_schedule": tfmot.sparsity.keras.PolynomialDecay(
            initial_sparsity=0.0,
            final_sparsity=0.5,
            begin_step=0,
            end_step=end_step
        )
    }

    studentQ = tfmot.sparsity.keras.prune_low_magnitude(
        qmodel,
        **pruning_params
    )

    return studentQ

### Compilación del modelo Student para KD y QAT

In [ ]:
studentQ = build_student(neurons_student, x_train)
studentQ.summary()

In [ ]:
distilled_student = Distiller(student=studentQ, teacher=modelQAT)
adam = Adam(learning_rate=lr)
train_labels = np.argmax(y_train, axis=1)

distilled_student.compile(
        optimizer=adam,
        metrics=[keras.metrics.SparseCategoricalAccuracy()],
        student_loss_fn=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        distillation_loss_fn=keras.losses.KLDivergence(),
        alpha=0.1,
        temperature=10,
    )

### Entrenamiento del modelo Student

In [ ]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_student_loss',
        factor=0.3,
        patience=3,
        verbose=1
    ),
    pruning_callbacks.UpdatePruningStep(),
]

history_student = distilled_student.fit(
    x=x_train,
    y=train_labels,
    batch_size=64,
    epochs=200,
    validation_split=0.2,
    callbacks=callbacks,
)

In [ ]:
print(history_student.history.keys())

plt.figure(figsize=(15,3))
plt.plot(history_student.history['sparse_categorical_accuracy'])
plt.plot(history_student.history['val_sparse_categorical_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()
plt.figure()

plt.figure(figsize=(15,3))
plt.plot(history_student.history['student_loss'])
plt.plot(history_student.history['val_student_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

In [ ]:
y_pred_probs = distilled_student.student.predict(dfTest)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(yTest, axis=1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Anomalo', 'Normal'])
disp.plot(cmap="Blues")
plt.title('Confusion Matrix - Student model Q, P, and KD (ECG PTB)')
plt.show()

### Prueba de predicción individual

`indexPrediction` es el índice de la señal en el conjunto de test. Nuestro `dfTest` tiene 1000 filas (500 por clase), así que cualquier índice entre 0 y 999 es válido.

In [ ]:
indexPrediction = 405

x_input = dfTest.iloc[indexPrediction]
y_label = yTest[indexPrediction]

inputPred = array([x_input])

y_pred = distilled_student.student.predict(inputPred)
print("> Input %s -> Predicted = %s | " % (y_label, y_pred))

plt.figure(figsize=(15,3))
plt.xlabel('Samples', fontsize=11)
plt.ylabel('Amplitude', fontsize=11)
plt.grid(True, alpha=1.0)
plt.plot(x_input.values,  label="Signal 1", color='navy', markersize=7, lw=1)

plt.title('Signal used for inference - Pulse: %s, Label: %s, Prediction: %s' % (indexPrediction, y_label, y_pred))

In [ ]:
indexPrediction = 200

x_input = dfTest.iloc[indexPrediction]
y_label = yTest[indexPrediction]

inputPred = array([x_input])

y_pred = distilled_student.student.predict(inputPred)
print("> Input %s -> Predicted = %s | " % (y_label, y_pred))

plt.figure(figsize=(15,3))
plt.xlabel('Samples', fontsize=11)
plt.ylabel('Amplitude', fontsize=11)
plt.grid(True, alpha=1.0)
plt.plot(x_input.values,  label="Signal 1", color='navy', markersize=7, lw=1)

plt.title('Signal used for inference - Pulse: %s, Label: %s, Prediction: %s' % (indexPrediction, y_label, y_pred))

### Guardar el modelo Student

**Este es el archivo que necesitas para P3** — descárgalo o súbelo a Drive/repo antes de cerrar la sesión.

In [ ]:
model_student = strip_pruning(distilled_student.student)
model_student.summary()

os.makedirs('models', exist_ok=True)
model_student.save('models/studentModel_ECG_PTB.h5')

from google.colab import files
files.download('models/studentModel_ECG_PTB.h5')

---
**Siguiente paso:** P3 — conversión con hls4ml y (fuera de Colab) síntesis en Vitis HLS / Vivado.